# 💬 Project 20: Multi-Class Intent Classifier with Out-Of-Distribution (OOD) Detection
### Enterprise NLP, Intent Routing & Mahalanobis Distance OOD Rejection

**Author:** Data Science Portfolio Team  
**Difficulty:** 🔴 Advanced  
**Domain:** Conversational AI & Enterprise NLP  

---
### Notebook Outline:
1. **Environment Setup**
2. **Customer Dialogue Queries Ingestion**
3. **TF-IDF & Text Embedding Pipeline**
4. **In-Domain Intent Classifier Training**
5. **Out-of-Distribution (OOD) Detection via Max Softmax Probability vs. Distance Thresholding**

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

warnings.filterwarnings('ignore')
print("Enterprise NLP workspace ready.")

In [ ]:
# Ingestion & In-Domain vs OOD Split
df = pd.read_csv("data/customer_support_intents.csv")
print(f"Total Customer Queries: {len(df)} | OOD Queries: {df['is_ood'].sum()}")

train_mask = df['is_ood'] == 0
train_df = df[train_mask]

vectorizer = TfidfVectorizer(max_features=500)
X_train = vectorizer.fit_transform(train_df['query_text'])
y_train = train_df['intent_label']

clf = LogisticRegression(random_state=42)
clf.fit(X_train, y_train)

# OOD Scoring on Full Dataset
X_all = vectorizer.transform(df['query_text'])
probs_all = clf.predict_proba(X_all)
max_probs = np.max(probs_all, axis=1)

# OOD detection AUROC: OOD queries should have lower max probability
ood_score = 1.0 - max_probs
auroc_ood = roc_auc_score(df['is_ood'], ood_score)
print(f"OOD Detection AUROC: {auroc_ood:.4f}")